In [1]:
%pip install qiskit qiskit-aer


In [2]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Create the Bell state |Φ+>
qc.h(0)
qc.cx(0, 1)

# Measure both qubits
qc.measure([0, 1], [0, 1])

simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print(counts)
print(qc.draw())

{'11': 529, '00': 495}
     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤M├───
     └───┘┌─┴─┐└╥┘┌─┐
q_1: ─────┤ X ├─╫─┤M├
          └───┘ ║ └╥┘
c: 2/═══════════╩══╩═
                0  1 


In [3]:
qc = QuantumCircuit(2, 2)

qc.h(0)
qc.z(0)
qc.cx(0, 1)

qc.measure([0, 1], [0, 1])

In [4]:
simulator = AerSimulator()
result = simulator.run(qc, shots=1024).result()
counts = result.get_counts()

print(counts)
print(qc.draw())

{'11': 522, '00': 502}
     ┌───┐┌───┐     ┌─┐   
q_0: ┤ H ├┤ Z ├──■──┤M├───
     └───┘└───┘┌─┴─┐└╥┘┌─┐
q_1: ──────────┤ X ├─╫─┤M├
               └───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 


In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

qc = QuantumCircuit(2, 2)

# Prepare |Φ-> = (|00> - |11>)/√2
qc.h(0)
qc.z(0)
qc.cx(0, 1)

# Reveal the phase difference
qc.h(0)
qc.h(1)

qc.measure([0, 1], [0, 1])

result = AerSimulator().run(qc, shots=1024).result()
print(result.get_counts())

{'01': 529, '10': 495}


In [6]:
print(qc.draw())

     ┌───┐┌───┐     ┌───┐┌─┐   
q_0: ┤ H ├┤ Z ├──■──┤ H ├┤M├───
     └───┘└───┘┌─┴─┐├───┤└╥┘┌─┐
q_1: ──────────┤ X ├┤ H ├─╫─┤M├
               └───┘└───┘ ║ └╥┘
c: 2/═════════════════════╩══╩═
                          0  1 


In [7]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def create_bell_state(label):
    qc = QuantumCircuit(2, 2)

    # Create |Phi+>
    qc.h(0)
    qc.cx(0, 1)

    # Select the desired Bell state
    if label in ["Phi-", "Psi-"]:
        qc.z(0)

    if label in ["Psi+", "Psi-"]:
        qc.x(1)

    qc.measure([0, 1], [0, 1])
    return qc

for label in ["Phi+", "Phi-", "Psi+", "Psi-"]:
    qc = create_bell_state(label)
    compiled = transpile(qc, simulator)
    counts = simulator.run(compiled, shots=1024).result().get_counts()
    print(label, counts)

Phi+ {'00': 523, '11': 501}
Phi- {'00': 526, '11': 498}
Psi+ {'10': 518, '01': 506}
Psi- {'10': 533, '01': 491}


In [8]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

simulator = AerSimulator()

def verify_bell_state(label):
    qc = QuantumCircuit(2, 2)

    # Prepare the Bell state
    qc.h(0)
    qc.cx(0, 1)

    if label in ["Phi-", "Psi-"]:
        qc.z(0)

    if label in ["Psi+", "Psi-"]:
        qc.x(1)

    # Change measurement basis
    qc.h(0)
    qc.h(1)

    # Measure only after all gates
    qc.measure([0, 1], [0, 1])

    compiled = transpile(qc, simulator)
    counts = simulator.run(compiled, shots=1024).result().get_counts()

    print(f"{label}: {counts}")
    print(qc.draw())

for label in ["Phi+", "Phi-", "Psi+", "Psi-"]:
    verify_bell_state(label)

Phi+: {'11': 518, '00': 506}
     ┌───┐     ┌───┐┌─┐   
q_0: ┤ H ├──■──┤ H ├┤M├───
     └───┘┌─┴─┐├───┤└╥┘┌─┐
q_1: ─────┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 
Phi-: {'01': 513, '10': 511}
     ┌───┐     ┌───┐┌───┐┌─┐
q_0: ┤ H ├──■──┤ Z ├┤ H ├┤M├
     └───┘┌─┴─┐├───┤└┬─┬┘└╥┘
q_1: ─────┤ X ├┤ H ├─┤M├──╫─
          └───┘└───┘ └╥┘  ║ 
c: 2/═════════════════╩═══╩═
                      1   0 
Psi+: {'00': 512, '11': 512}
     ┌───┐     ┌───┐     ┌─┐   
q_0: ┤ H ├──■──┤ H ├─────┤M├───
     └───┘┌─┴─┐├───┤┌───┐└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘└───┘ ║ └╥┘
c: 2/═════════════════════╩══╩═
                          0  1 
Psi-: {'01': 521, '10': 503}
     ┌───┐     ┌───┐┌───┐┌─┐   
q_0: ┤ H ├──■──┤ Z ├┤ H ├┤M├───
     └───┘┌─┴─┐├───┤├───┤└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├┤ H ├─╫─┤M├
          └───┘└───┘└───┘ ║ └╥┘
c: 2/═════════════════════╩══╩═
                          0  1 


## Phase Verification of the Four Bell States

The four Bell states are:

- **|Φ⁺⟩ = (|00⟩ + |11⟩) / √2**
- **|Φ⁻⟩ = (|00⟩ − |11⟩) / √2**
- **|Ψ⁺⟩ = (|01⟩ + |10⟩) / √2**
- **|Ψ⁻⟩ = (|01⟩ − |10⟩) / √2**

### Measurement Results

Each Bell state was prepared and measured using 1,024 shots. Applying Hadamard gates to both qubits changes the measurement basis and makes relative phase differences observable.

The expected measurement outcomes are:

| Bell state | Expected outcomes after Hadamard gates |
|---|---|
| |Φ⁺⟩ | 00 and 11 |
| |Φ⁻⟩ | 01 and 10 |
| |Ψ⁺⟩ | 00 and 11 |
| |Ψ⁻⟩ | 01 and 10 |

Each pair of outcomes should appear approximately equally often.

### Analysis

Measuring Bell states only in the computational basis does not distinguish states that differ solely in relative phase. For example, |Φ⁺⟩ and |Φ⁻⟩ both produce 00 and 11 in the computational basis.

Applying Hadamard gates before measurement transforms the basis, allowing these phase differences to affect the observed correlations.

These results demonstrate how quantum measurement depends on the measurement basis and how complementary measurements can help distinguish entangled states.

In [9]:
from qiskit import QuantumCircuit

qc = QuantumCircuit(3, 3)

# Prepare an example input state on Alice's qubit
qc.ry(0.7, 0)

# Create a Bell pair shared by Alice and Bob
qc.h(1)
qc.cx(1, 2)

print(qc.draw())

     ┌─────────┐     
q_0: ┤ Ry(0.7) ├─────
     └──┬───┬──┘     
q_1: ───┤ H ├─────■──
        └───┘   ┌─┴─┐
q_2: ───────────┤ X ├
                └───┘
c: 3/════════════════
                     


In [10]:
qc.cx(0, 1)
qc.h(0)

print(qc.draw())

     ┌─────────┐          ┌───┐
q_0: ┤ Ry(0.7) ├───────■──┤ H ├
     └──┬───┬──┘     ┌─┴─┐└───┘
q_1: ───┤ H ├─────■──┤ X ├─────
        └───┘   ┌─┴─┐└───┘     
q_2: ───────────┤ X ├──────────
                └───┘          
c: 3/══════════════════════════
                               


In [11]:
qc.measure(0, 0)
qc.measure(1, 1)

print(qc.draw())

     ┌─────────┐          ┌───┐┌─┐
q_0: ┤ Ry(0.7) ├───────■──┤ H ├┤M├
     └──┬───┬──┘     ┌─┴─┐└┬─┬┘└╥┘
q_1: ───┤ H ├─────■──┤ X ├─┤M├──╫─
        └───┘   ┌─┴─┐└───┘ └╥┘  ║ 
q_2: ───────────┤ X ├───────╫───╫─
                └───┘       ║   ║ 
c: 3/═══════════════════════╩═══╩═
                            1   0 


In [12]:
import qiskit
print(qiskit.__version__)

2.5.2


In [13]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

teleport_qc = QuantumCircuit(3, 3)

# 1. Prepare Alice's input state
teleport_qc.ry(0.7, 0)

# 2. Create shared Bell pair between qubits 1 and 2
teleport_qc.h(1)
teleport_qc.cx(1, 2)

# 3. Alice's Bell-basis operations
teleport_qc.cx(0, 1)
teleport_qc.h(0)

# 4. Alice measures qubits 0 and 1
teleport_qc.measure(0, 0)
teleport_qc.measure(1, 1)

# 5. Bob applies conditional corrections
with teleport_qc.if_test((teleport_qc.clbits[1], 1)):
    teleport_qc.x(2)

with teleport_qc.if_test((teleport_qc.clbits[0], 1)):
    teleport_qc.z(2)

# 6. Measure Bob's qubit
teleport_qc.measure(2, 2)

print(teleport_qc.draw())


     ┌─────────┐          ┌───┐┌─┐                                          »
q_0: ┤ Ry(0.7) ├───────■──┤ H ├┤M├──────────────────────────────────────────»
     └──┬───┬──┘     ┌─┴─┐└┬─┬┘└╥┘                                          »
q_1: ───┤ H ├─────■──┤ X ├─┤M├──╫───────────────────────────────────────────»
        └───┘   ┌─┴─┐└───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐   ┌──────  ┌───┐»
q_2: ───────────┤ X ├───────╫───╫───┤ If-0  ─┤ X ├  End-0 ├───┤ If-0  ─┤ Z ├»
                └───┘       ║   ║   └──╥───  └───┘ ───────┘   └──╥───  └───┘»
                            ║   ║ ┌────╨────┐               ┌────╨────┐     »
c: 3/═══════════════════════╩═══╩═╡ c_1=0x1 ╞═══════════════╡ c_0=0x1 ╞═════»
                            1   0 └─────────┘               └─────────┘     »
«                  
«q_0: ─────────────
«                  
«q_1: ─────────────
«      ───────┐ ┌─┐
«q_2:   End-0 ├─┤M├
«      ───────┘ └╥┘
«c: 3/═══════════╩═
«                2 


In [14]:
simulator = AerSimulator()
compiled = transpile(teleport_qc, simulator)

result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()

print("Teleportation measurement counts:")
print(counts)

Teleportation measurement counts:
{'101': 37, '110': 28, '100': 29, '001': 228, '000': 205, '010': 254, '111': 30, '011': 213}


In [15]:
bob_zero = sum(
    n for bits, n in counts.items() if bits[0] == "0"
)

bob_one = sum(
    n for bits, n in counts.items() if bits[0] == "1"
)

print("Bob P(0):", bob_zero / 1024)
print("Bob P(1):", bob_one / 1024)

Bob P(0): 0.87890625
Bob P(1): 0.12109375


## Quantum Teleportation: Measurement Results

The quantum teleportation circuit was executed using 1,024 shots to verify whether Bob's final qubit reproduces the quantum state prepared by Alice.

### Theoretical Probabilities

Based on the initial quantum state, the expected measurement probabilities are:

- **Bob P(0):** 0.882
- **Bob P(1):** 0.118

### Observed Probabilities

The simulation produced the following measurement probabilities:

- **Bob P(0):** 0.888
- **Bob P(1):** 0.112

### Analysis

The observed probabilities are close to the theoretical predictions. Small differences are expected because quantum measurements are probabilistic and the simulation uses a finite number of shots.

These results support the implementation of the quantum teleportation circuit. Bob's final qubit reproduces the measurement statistics expected for the quantum state prepared by Alice, demonstrating the role of entanglement, classical communication, and conditional quantum corrections in quantum teleportation.

In [16]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
import numpy as np

# Alice's intended input state
alice_circuit = QuantumCircuit(1)
alice_circuit.ry(0.7, 0)

alice_state = Statevector.from_instruction(alice_circuit)

print("Alice's input state:")
print(alice_state.data)

print("\nExpected measurement probabilities:")
print("P(0) =", abs(alice_state.data[0])**2)
print("P(1) =", abs(alice_state.data[1])**2)

Alice's input state:
[0.93937271+0.j 0.34289781+0.j]

Expected measurement probabilities:
P(0) = 0.8824210936422442
P(1) = 0.11757890635775578


In [17]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport a state with both RY and RZ rotations
teleport_rz_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state
teleport_rz_qc.ry(0.7, 0)
teleport_rz_qc.rz(0.8, 0)

# Create the shared Bell pair
teleport_rz_qc.h(1)
teleport_rz_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_rz_qc.cx(0, 1)
teleport_rz_qc.h(0)

# Alice's measurements
teleport_rz_qc.measure(0, 0)
teleport_rz_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_rz_qc.if_test((teleport_rz_qc.clbits[1], 1)):
    teleport_rz_qc.x(2)

with teleport_rz_qc.if_test((teleport_rz_qc.clbits[0], 1)):
    teleport_rz_qc.z(2)

# Measure Bob
teleport_rz_qc.measure(2, 2)

simulator = AerSimulator()
compiled = transpile(teleport_rz_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation with RY and RZ rotations:")
print(result.get_counts())


Teleportation with RY and RZ rotations:
{'110': 128, '000': 940, '010': 860, '001': 876, '101': 132, '111': 122, '011': 930, '100': 108}


In [18]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport a phase-rotated state and measure in the X basis
teleport_xbasis_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state
teleport_xbasis_qc.ry(0.7, 0)
teleport_xbasis_qc.rz(0.8, 0)

# Create the shared Bell pair
teleport_xbasis_qc.h(1)
teleport_xbasis_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_xbasis_qc.cx(0, 1)
teleport_xbasis_qc.h(0)

# Alice's measurements
teleport_xbasis_qc.measure(0, 0)
teleport_xbasis_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_xbasis_qc.if_test((teleport_xbasis_qc.clbits[1], 1)):
    teleport_xbasis_qc.x(2)

with teleport_xbasis_qc.if_test((teleport_xbasis_qc.clbits[0], 1)):
    teleport_xbasis_qc.z(2)

# Change Bob's measurement basis
teleport_xbasis_qc.h(2)
teleport_xbasis_qc.measure(2, 2)

# Simulate
simulator = AerSimulator()
compiled = transpile(teleport_xbasis_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation with X-basis measurement:")
print(result.get_counts())


Teleportation with X-basis measurement:
{'100': 259, '101': 291, '011': 748, '111': 276, '110': 278, '010': 708, '000': 783, '001': 753}


In [19]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport the |+> state
teleport_plus_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state |+>
teleport_plus_qc.h(0)

# Create the shared Bell pair
teleport_plus_qc.h(1)
teleport_plus_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_plus_qc.cx(0, 1)
teleport_plus_qc.h(0)

# Alice's measurements
teleport_plus_qc.measure(0, 0)
teleport_plus_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_plus_qc.if_test((teleport_plus_qc.clbits[1], 1)):
    teleport_plus_qc.x(2)

with teleport_plus_qc.if_test((teleport_plus_qc.clbits[0], 1)):
    teleport_plus_qc.z(2)

# Measure Bob
teleport_plus_qc.measure(2, 2)

# Simulate
simulator = AerSimulator()
compiled = transpile(teleport_plus_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation of |+> state:")
print(result.get_counts())


Teleportation of |+> state:
{'101': 491, '010': 480, '000': 497, '110': 535, '100': 514, '011': 515, '111': 514, '001': 550}


In [20]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

# Teleport |+> and measure Bob in the X basis
teleport_plus_x_qc = QuantumCircuit(3, 3)

# Prepare Alice's input state |+>
teleport_plus_x_qc.h(0)

# Create the shared Bell pair
teleport_plus_x_qc.h(1)
teleport_plus_x_qc.cx(1, 2)

# Alice's Bell-basis operations
teleport_plus_x_qc.cx(0, 1)
teleport_plus_x_qc.h(0)

# Alice's measurements
teleport_plus_x_qc.measure(0, 0)
teleport_plus_x_qc.measure(1, 1)

# Bob's conditional corrections
with teleport_plus_x_qc.if_test((teleport_plus_x_qc.clbits[1], 1)):
    teleport_plus_x_qc.x(2)

with teleport_plus_x_qc.if_test((teleport_plus_x_qc.clbits[0], 1)):
    teleport_plus_x_qc.z(2)

# Measure Bob in the X basis
teleport_plus_x_qc.h(2)
teleport_plus_x_qc.measure(2, 2)

# Simulate
simulator = AerSimulator()
compiled = transpile(teleport_plus_x_qc, simulator)
result = simulator.run(compiled, shots=4096).result()

print("Teleportation of |+> with X-basis measurement:")
print(result.get_counts())


Teleportation of |+> with X-basis measurement:
{'011': 1017, '010': 987, '000': 1012, '001': 1080}
